# Scala basics

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/scala/scala.ipynb)

Scala is a statically typed JVM language that mixes object-oriented and functional programming. Spark, Kafka (originally) and Flink's Scala API are written in it, so data engineers meet it when reading Spark internals, writing typed Dataset jobs, or maintaining JVM pipelines. Scala needs a JVM and a compiler, so Scala code appears in fenced snippets (Scala 3 syntax, with Scala 2 notes where they differ), and the concepts that matter for distributed jobs are reproduced in runnable Python next to them.

## Why data engineers learn Scala

+ Spark's native API: typed `Dataset[T]`, custom aggregators, Catalyst extensions and connectors are Scala-first; reading Spark source and stack traces needs it.
+ JVM-native logic avoids the Python worker round-trip that Python UDFs pay (serialization between the JVM and Python processes).
+ Compile-time types catch schema and refactoring mistakes before a job runs for an hour.
+ Kafka Streams, Flink, Akka/Pekko and many connectors live on the JVM.

The trade-offs: slower compiles, a smaller hiring pool than Python, binary compatibility between Scala versions (2.12, 2.13 and 3 artifacts are not interchangeable; Spark 3.5 ships for 2.12 and 2.13, Spark 4 for 2.13), and many teams now do most Spark work in PySpark or SQL, where the DataFrame API compiles to the same plan.

## Values, types and expressions

`val` is an immutable binding; `var` is reassignable. Prefer `val` and immutable collections: they are safe to share across threads and Futures, and in Spark and Flink, closures are serialized and shipped to executors, where a captured mutable variable is just a copy.

```scala
val rate: Double = 0.2          // cannot be reassigned
var retries = 0                 // reassignable; keep local and rare
val total = 42                  // type inferred as Int
val name = s"orders_$total"     // s-interpolator; f"$rate%.2f" formats; raw"\d+" skips escapes

// Everything is an expression: if and match return values
val tier = if total > 100 then "large" else "small"
```

Basic types are `Int`, `Long`, `Double`, `BigDecimal` (use it for money), `Boolean`, `Char`, `String`, `Unit` (no meaningful value, like `void`), `Any` (top type) and `Nothing` (bottom type, the type of `throw`). Local types are inferred; write explicit types on public methods so APIs do not change by accident.

## Functions, higher-order functions and currying

```scala
def netAmount(gross: BigDecimal, taxRate: BigDecimal = 0.2): BigDecimal = gross * (1 - taxRate)

val double: Int => Int = x => x * 2              // function value
List(1, 2, 3).map(_ * 2)                          // placeholder syntax
def applyTwice(f: Int => Int, x: Int): Int = f(f(x))

def withRetry[A](attempts: Int)(body: => A): A =  // curried, with a by-name block parameter
  try body catch case e: Exception if attempts > 1 => withRetry(attempts - 1)(body)

val result = withRetry(3) { fetchFromApi() }
```

Methods (`def`) belong to a class or object; functions are values of type `A => B` and can be passed around. Currying (multiple parameter lists) enables block syntax as above, helps type inference, and lets you partially apply arguments (`val addTax = netAmount(_, 0.25)`). A by-name parameter (`body: => A`) is evaluated each time it is used rather than once at the call, which is what makes `withRetry` re-run the block.

## Case classes, pattern matching and sealed traits

A **case class** is an immutable record with structural equality, `hashCode`, `toString`, `copy`, and pattern-matching support generated for you. In Spark, case classes define the schema of a typed `Dataset`.

```scala
final case class Order(id: Long, customerId: String, amount: BigDecimal, status: String)

val paid = order.copy(status = "PAID")     // new instance; original unchanged

sealed trait Event                          // all subtypes must be in this file
final case class Created(orderId: Long, amount: BigDecimal) extends Event
final case class Refunded(orderId: Long, amount: BigDecimal) extends Event
case object Heartbeat extends Event

def revenueDelta(event: Event): BigDecimal = event match
  case Created(_, amount)              => amount
  case Refunded(_, amount) if amount > 0 => -amount
  case Refunded(_, _)                  => 0
  case Heartbeat                       => 0
```

Because `Event` is `sealed`, the compiler warns if a `match` misses a case: add a new event type and every handler that forgot it fails the build. That is the main reason to model messages and states as sealed hierarchies (Scala 3 `enum` is the shorthand). Patterns can destructure nested data, bind variables, use guards (`if`), match types (`case s: String`) and use custom extractors (an `unapply` method). Python 3.10's `match` and dataclasses give a close runnable analogue:

### Where you'll use it

- **Reading and extending Spark:** Spark, Kafka Streams and many internal data libraries are written in Scala, so bugs and features are easier to follow when you read the source.

### Where you'll use it

- **Typed event handling:** a sealed trait lists every event type, and the compiler warns when a handler forgets one.

In [ ]:
from dataclasses import dataclass, replace
from decimal import Decimal


@dataclass(frozen=True)
class Created:
    order_id: int
    amount: Decimal


@dataclass(frozen=True)
class Refunded:
    order_id: int
    amount: Decimal


@dataclass(frozen=True)
class Heartbeat:
    pass


Event = Created | Refunded | Heartbeat


def revenue_delta(event: Event) -> Decimal:
    match event:
        case Created(amount=amount):
            return amount
        case Refunded(amount=amount) if amount > 0:
            return -amount
        case Refunded() | Heartbeat():
            return Decimal(0)
    raise TypeError(f"unhandled event {event!r}")  # Scala's sealed check makes this a compile error


events = [Created(1, Decimal("40.00")), Refunded(1, Decimal("15.00")), Heartbeat(), Created(2, Decimal("10.00"))]
net = sum((revenue_delta(e) for e in events), Decimal(0))
first = events[0]
adjusted = replace(first, amount=Decimal("45.00"))  # like case class .copy
print("net revenue:", net)
assert net == Decimal("35.00")
assert first.amount == Decimal("40.00") and adjusted.amount == Decimal("45.00")
assert Created(1, Decimal("40.00")) == first  # structural equality, like case classes

## Traits, classes and objects

+ A **trait** is an interface that can carry implementation and state; a class can mix in several (`class Job extends Reader with Writer with Logging`). Linearization resolves conflicts: the rightmost trait's method wins and `super` calls chain leftwards.
+ An **abstract class** can take constructor parameters (traits can too in Scala 3) and is friendlier to Java callers.
+ An **object** is a singleton: the home of constants, factories and `main`. A **companion object** shares the name and file of a class and can access its private members; its `apply` makes `Order(...)` work without `new`, and `unapply` powers pattern matching.

```scala
trait Source[A]:
  def read(): Iterator[A]

trait Logging:
  protected def log(msg: String): Unit = println(s"[${getClass.getSimpleName}] $msg")

final class CsvSource(path: String) extends Source[Order] with Logging:
  def read(): Iterator[Order] = { log(s"reading $path"); Iterator.empty }

final case class TableName(schema: String, table: String)
object TableName:
  def parse(qualified: String): Either[String, TableName] = qualified.split('.') match
    case Array(s, t) => Right(TableName(s, t))
    case _           => Left(s"expected schema.table, got $qualified")
```

A Spark-specific trap: a trait or object that holds a `SparkSession`, a database connection or a logger instance gets captured in closures and fails with `Task not serializable`. Keep such resources out of the objects your transformations reference.

## Collections and combinators

Immutable collections are the default: `List` (linked list, fast prepend), `Vector` (fast indexed access and append), `Map`, `Set`, `Seq` (general sequence interface). Mutable versions live in `scala.collection.mutable` (`ArrayBuffer`, `HashMap`) and are fine as local builders. `Array` is a JVM array: mutable, fast, and what Spark uses internally.

```scala
val orders: List[Order] = loadOrders()

orders.filter(_.status == "PAID").map(_.amount).sum
orders.groupBy(_.customerId).view.mapValues(_.map(_.amount).sum).toMap
orders.foldLeft(BigDecimal(0))((acc, o) => acc + o.amount)
orders.partition(_.amount > 100)                   // (big, small)
orders.flatMap(o => o.lineItems)                   // map then flatten
orders.collect { case Order(id, _, amt, "PAID") if amt > 0 => id }   // filter + map with a pattern
orders.sortBy(o => (o.customerId, -o.amount.toDouble))
orders.sliding(3).map(window => window.map(_.amount).sum / 3)
orders.iterator.grouped(500).foreach(batch => writeBatch(batch))     // bounded memory
```

The same combinators exist on Spark `Dataset`s (`map`, `filter`, `flatMap`, `groupByKey`, `reduce`), which is why Scala collection code reads like Spark code. A Python version of `foldLeft`, `groupBy` and `partition`:

In [ ]:
from collections import defaultdict
from functools import reduce

orders = [
    {"id": 1, "customer": "c1", "amount": Decimal("120.00"), "status": "PAID"},
    {"id": 2, "customer": "c2", "amount": Decimal("30.00"), "status": "PENDING"},
    {"id": 3, "customer": "c1", "amount": Decimal("80.00"), "status": "PAID"},
    {"id": 4, "customer": "c3", "amount": Decimal("200.00"), "status": "PAID"},
]

# orders.foldLeft(Summary())((acc, o) => ...): one pass, immutable accumulator
summary = reduce(
    lambda acc, o: {"count": acc["count"] + 1, "total": acc["total"] + o["amount"],
                    "max": max(acc["max"], o["amount"])},
    orders,
    {"count": 0, "total": Decimal(0), "max": Decimal(0)},
)

# orders.groupBy(_.customer).view.mapValues(_.map(_.amount).sum)
by_customer: dict[str, Decimal] = defaultdict(Decimal)
for order in orders:
    by_customer[order["customer"]] += order["amount"]

# orders.partition(_.amount > 100)
big = [o["id"] for o in orders if o["amount"] > 100]
small = [o["id"] for o in orders if o["amount"] <= 100]

# orders.collect { case o if o.status == "PAID" => o.id }
paid_ids = [o["id"] for o in orders if o["status"] == "PAID"]

print(summary, dict(by_customer), (big, small), paid_ids)
assert summary == {"count": 4, "total": Decimal("430.00"), "max": Decimal("200.00")}
assert by_customer["c1"] == Decimal("200.00")
assert (big, small) == ([1, 4], [2, 3]) and paid_ids == [1, 3, 4]

## For-comprehensions

A `for` with `yield` is syntax sugar for `flatMap`, `map` and `withFilter`, so it works on any type that defines them: collections, `Option`, `Either`, `Try`, `Future`, Spark `Dataset`s.

```scala
for
  a <- as
  b <- bs
  if a < b
yield (a, b)

// desugars to
as.flatMap(a => bs.withFilter(b => a < b).map(b => (a, b)))
```

With `Option` or `Either`, the same syntax becomes short-circuiting error handling: the first `None` or `Left` stops the chain.

In [ ]:
def flat_map(xs, f):
    return [y for x in xs for y in f(x)]


a_values, b_values = [1, 2, 3], [2, 3]

comprehension = [(a, b) for a in a_values for b in b_values if a < b]
desugared = flat_map(a_values, lambda a: [(a, b) for b in b_values if a < b])
print(comprehension)
assert comprehension == desugared == [(1, 2), (1, 3), (2, 3)]

## Option, Either and Try instead of null and exceptions

+ `Option[A]` is `Some(a)` or `None`: a value that may be missing. Wrap nullable Java results with `Option(javaCall())`.
+ `Either[E, A]` is `Left(error)` or `Right(value)`: a computation that can fail with a typed reason. Right-biased, so `map` and `flatMap` work on the success side.
+ `Try[A]` is `Success(a)` or `Failure(exception)`: wraps code that throws.

```scala
def configValue(key: String): Option[String] = sys.env.get(key)

def parsePort(raw: String): Either[String, Int] =
  raw.toIntOption.filter(p => p > 0 && p < 65536).toRight(s"invalid port: $raw")

val endpoint: Either[String, String] = for
  host <- configValue("WAREHOUSE_HOST").toRight("WAREHOUSE_HOST not set")
  port <- configValue("WAREHOUSE_PORT").toRight("WAREHOUSE_PORT not set").flatMap(parsePort)
yield s"$host:$port"

val parsed: Try[Int] = Try("12x".toInt)        // Failure(NumberFormatException)
parsed.getOrElse(0)
```

Avoid `.get` on `Option` and `Try`: it throws on the empty case and defeats the purpose. Validating every config key up front and reporting all errors (rather than the first) is what libraries like Cats `Validated` add. The short-circuit behaviour in Python:

### Where you'll use it

- **Parsing with errors as values:** an `Either` per record separates good rows from rejects without a try/catch around the whole file.

In [ ]:
from typing import Callable


class Left:
    def __init__(self, error: str):
        self.error = error

    def flat_map(self, _f: Callable):
        return self  # the first failure short-circuits the chain

    def __repr__(self):
        return f"Left({self.error!r})"


class Right:
    def __init__(self, value):
        self.value = value

    def flat_map(self, f: Callable):
        return f(self.value)

    def __repr__(self):
        return f"Right({self.value!r})"


def config_value(env: dict, key: str):
    return Right(env[key]) if key in env else Left(f"{key} not set")


def parse_port(raw: str):
    return Right(int(raw)) if raw.isdigit() and 0 < int(raw) < 65536 else Left(f"invalid port: {raw}")


def endpoint(env: dict):
    return config_value(env, "WAREHOUSE_HOST").flat_map(
        lambda host: config_value(env, "WAREHOUSE_PORT").flat_map(parse_port).flat_map(
            lambda port: Right(f"{host}:{port}")))


for env in [{"WAREHOUSE_HOST": "wh", "WAREHOUSE_PORT": "5439"}, {"WAREHOUSE_PORT": "5439"},
            {"WAREHOUSE_HOST": "wh", "WAREHOUSE_PORT": "99999"}]:
    print(env, "->", endpoint(env))
assert endpoint({"WAREHOUSE_HOST": "wh", "WAREHOUSE_PORT": "5439"}).value == "wh:5439"
assert endpoint({"WAREHOUSE_PORT": "5439"}).error == "WAREHOUSE_HOST not set"
assert endpoint({"WAREHOUSE_HOST": "wh", "WAREHOUSE_PORT": "99999"}).error == "invalid port: 99999"

## Implicits, givens and extension methods

Scala 2 `implicit` covered several features; Scala 3 splits them:

+ **Given instances and `using` clauses** (Scala 2: implicit values and parameters) pass context such as an `ExecutionContext`, a `SparkSession` or a type-class instance without threading it through every call.
+ **Extension methods** (Scala 2: implicit classes) add methods to existing types.
+ **Implicit conversions** still exist but must be enabled explicitly; avoid them.

Spark relies on this: `import spark.implicits._` brings `Encoder` instances for case classes and primitives into scope (needed by `.as[Order]` and `Dataset.map`) and the `$"col"` syntax.

```scala
trait Show[A]:
  def show(a: A): String

given Show[Order] with
  def show(o: Order): String = s"Order(${o.id}, ${o.amount})"

def render[A](items: List[A])(using s: Show[A]): String = items.map(s.show).mkString("\n")

extension (df: DataFrame)
  def withIngestDate(date: LocalDate): DataFrame = df.withColumn("ingest_date", lit(date.toString))
```

Overusing implicits makes code hard to follow ("where does this value come from?"); keep them for genuine context and type classes.

## Generics and variance

```scala
def firstOr[A](xs: List[A], default: A): A = xs.headOption.getOrElse(default)

def maxBy[A, B: Ordering](xs: Seq[A])(f: A => B): Option[A] =   // context bound: needs an Ordering[B]
  xs.maxByOption(f)

def writeAll[T <: Event](events: Seq[T]): Unit = ()             // upper bound
```

Variance describes subtyping of containers: `List[+A]` is **covariant** (a `List[Created]` is a `List[Event]`), function parameters are **contravariant** (`Event => Unit` can be used where `Created => Unit` is expected), and mutable containers like `Array[A]` are **invariant**. On the JVM, type parameters are **erased** at runtime, so `case xs: List[String]` cannot check the element type; Spark's `Encoder` and Scala's `ClassTag` exist to carry that information.

## Lazy values and by-name parameters

`lazy val` computes on first access and caches the result (thread-safely): useful for expensive clients or config that may not be needed. In Spark, a `lazy val` inside an `object` is a common way to create one non-serializable client per executor JVM instead of shipping it from the driver.

```scala
object WarehouseClient:
  lazy val pool: ConnectionPool = ConnectionPool.create(sys.env("WAREHOUSE_URL"))  // once per JVM

df.foreachPartition { rows =>
  val conn = WarehouseClient.pool.borrow()   // initialized on the executor, not serialized
  try rows.grouped(500).foreach(batch => conn.insert(batch)) finally conn.close()
}
```

## Futures and concurrency

A `Future[A]` runs on an `ExecutionContext` (a thread pool) and completes with a value or an exception. Compose with `map`, `flatMap`, `for`, `Future.sequence` and `Future.traverse`; avoid `Await.result` except at the edge of the program. Blocking I/O inside a Future should be wrapped in `blocking { ... }` or run on a dedicated pool so it does not starve the default one. Bound parallelism when fanning out (process batches of N) so a list of 10,000 tables does not open 10,000 connections. Effect libraries (Cats Effect, ZIO) and streams (fs2, Akka/Pekko Streams) add cancellation, resource safety and backpressure.

```scala
import scala.concurrent.{ExecutionContext, Future}
import scala.concurrent.duration.*

given ExecutionContext = ExecutionContext.fromExecutor(java.util.concurrent.Executors.newFixedThreadPool(8))

def copyTable(name: String): Future[Long] = Future { blocking { warehouse.copy(name) } }

val total: Future[Long] = Future.traverse(tables)(copyTable).map(_.sum)
```

## Java interoperability

Scala compiles to JVM bytecode and calls Java libraries directly (JDBC, Kafka clients, AWS SDK, Hadoop). Convert collections with `scala.jdk.CollectionConverters._` (`javaList.asScala`, `scalaSeq.asJava`), wrap nullable returns in `Option(...)`, and remember checked exceptions are not enforced. Annotations such as `@throws` and `@varargs` help Java callers of Scala code.

## Build: sbt, dependencies and tests

```scala
// build.sbt
ThisBuild / scalaVersion := "2.13.15"     // must match the Spark distribution's Scala version
ThisBuild / organization := "com.acme"

lazy val jobs = (project in file("."))
  .settings(
    name := "orders-jobs",
    libraryDependencies ++= Seq(
      "org.apache.spark" %% "spark-sql" % "3.5.3" % Provided,   // %% appends _2.13
      "org.scalatest"    %% "scalatest" % "3.2.19" % Test,
    ),
    assembly / assemblyMergeStrategy := {
      case PathList("META-INF", _*) => MergeStrategy.discard
      case _                        => MergeStrategy.first
    },
  )
```

+ `%%` appends the Scala binary version to the artifact name; mixing `_2.12` and `_2.13` artifacts fails at runtime with `NoSuchMethodError`.
+ Mark Spark as `Provided`: the cluster supplies it, and bundling it bloats the JAR and causes version conflicts. `sbt assembly` builds the fat JAR for `spark-submit`; shade conflicting libraries (Jackson, Guava, protobuf).
+ Tests: ScalaTest or MUnit, with a local `SparkSession` (`master("local[2]")`, few shuffle partitions) for transformation tests.

### Where you'll use it

- **Concurrent API calls:** a service fetches several enrichment APIs in parallel and combines the results.

## Scala for Spark

```scala
import org.apache.spark.sql.{Dataset, SparkSession}
import org.apache.spark.sql.functions.*

final case class Order(orderId: Long, customerId: String, amount: Double, status: String)
final case class CustomerRevenue(customerId: String, revenue: Double)

object DailyRevenue:
  def transform(orders: Dataset[Order])(using spark: SparkSession): Dataset[CustomerRevenue] =
    import spark.implicits.*
    orders
      .filter($"status" === "PAID")                       // Column expression: Catalyst can push it down
      .groupBy($"customerId")
      .agg(sum($"amount").as("revenue"))
      .as[CustomerRevenue]

  def main(args: Array[String]): Unit =
    given spark: SparkSession = SparkSession.builder().appName("daily-revenue").getOrCreate()
    import spark.implicits.*
    val orders = spark.read.parquet(args(0)).as[Order]
    transform(orders).write.mode("overwrite").parquet(args(1))
```

Idioms that matter:

+ **Prefer Column expressions and built-in functions over typed lambdas and UDFs.** `ds.filter(_.status == "PAID")` is a black box to Catalyst: it deserializes every row into an object, blocks predicate pushdown and column pruning, and defeats whole-stage code generation. `filter($"status" === "PAID")` does not. Use the typed API where type safety matters more than the last bit of speed, and Spark SQL functions for hot paths.
+ **Keep transformations as pure functions** `Dataset[A] => Dataset[B]`, composed with `.transform(...)` and unit-tested on small local Datasets.
+ **Everything a closure references must be serializable.** Referencing a field of an enclosing class captures the whole instance (and its `SparkSession`, connections and loggers); copy the field into a local `val` first. Create clients per partition (`mapPartitions`, `foreachPartition`) or via a `lazy val` in an object.
+ **Driver-side variables are not shared.** A `var counter` incremented inside `foreach` changes each executor's deserialized copy; the driver still sees zero. Use accumulators for metrics or, better, aggregate.
+ **Avoid `collect()` on large data**, and prefer `reduceByKey` / `aggregateByKey` over `groupByKey` in RDD code.
+ **Allocation matters in hot loops.** Copying a case class per row per step creates garbage; in tight per-partition loops, a local mutable builder or `while` loop is acceptable as long as it does not escape the function.

The closure-capture bug, reproduced with `pickle` standing in for Spark's closure serialization: each "executor" gets its own copy of the captured state.

### Where you'll use it

- **Production Spark jobs:** typed `Dataset` APIs and case classes give compile-time checks on job code that would otherwise fail at runtime.

In [ ]:
import pickle


class DroppedCounter:
    def __init__(self):
        self.dropped = 0


driver_counter = DroppedCounter()
partitions = [[{"amount": 10}, {"amount": -1}], [{"amount": -5}, {"amount": 7}, {"amount": -2}]]


def run_task(serialized_closure_state: bytes, rows: list[dict]) -> list[dict]:
    counter = pickle.loads(serialized_closure_state)  # what an executor receives: a copy
    kept = []
    for row in rows:
        if row["amount"] < 0:
            counter.dropped += 1  # mutates the executor's copy only
        else:
            kept.append(row)
    return kept


kept_rows = [row for part in partitions for row in run_task(pickle.dumps(driver_counter), part)]
print("driver sees dropped =", driver_counter.dropped)
assert driver_counter.dropped == 0 and len(kept_rows) == 2

# Fix: return the information as data and aggregate it, like a Spark aggregation or accumulator.
def run_task_tagged(rows: list[dict]) -> tuple[list[dict], int]:
    kept = [row for row in rows if row["amount"] >= 0]
    return kept, len(rows) - len(kept)


results = [run_task_tagged(part) for part in partitions]
dropped_total = sum(dropped for _, dropped in results)
print("aggregated dropped =", dropped_total)
assert dropped_total == 3

### Immutable copies versus local builders

Immutability is the default for correctness, but copying every record at every step costs allocations and GC time on billions of rows. The usual compromise: immutable data at function boundaries, a mutable builder confined inside one function or partition.

In [ ]:
records = [{"id": i, "status": "PENDING"} for i in range(5000)]


def pure_update(rows: list[dict]) -> tuple[list[dict], int]:
    out = [{**row, "status": "PAID"} for row in rows]  # like case class .copy per row
    return out, len(out)


def mutable_update(rows: list[dict]) -> list[dict]:
    for row in rows:
        row["status"] = "PAID"  # in place: zero allocations, but every caller sees the change
    return rows


paid, allocations = pure_update(records)
assert allocations == 5000 and records[0]["status"] == "PENDING" and paid[0]["status"] == "PAID"

same = mutable_update(records)
assert same[0] is records[0] and records[0]["status"] == "PAID"
print(f"pure update allocated {allocations} new records; in-place update allocated none but mutated the input")

## Scala beyond Spark

+ **Kafka**: Scala services consume with the Java client (or fs2-kafka, Alpakka Kafka) and commit offsets after writing, for at-least-once landing of raw events.
+ **Flink**: the Scala API is deprecated in recent Flink versions; new Flink jobs in JVM languages are usually written against the Java API (callable from Scala) or Flink SQL.
+ **Shared libraries**: one Scala module of pure transformation functions used by both batch (Spark) and streaming (Structured Streaming or Flink) jobs.

## Common pitfalls

+ Scala binary version mismatch between your JAR and the cluster's Spark.
+ Bundling Spark or Hadoop in the fat JAR instead of marking them `Provided`.
+ `Task not serializable` from closures that capture an outer class, a `SparkSession` or a client.
+ Typed lambdas and UDFs on hot paths, blocking Catalyst optimizations.
+ `Option.get`, `null` from Java APIs, and `Await.result` sprinkled through code.
+ Implicit magic that hides where values come from.
+ `var` counters updated inside distributed operations.

Questions to check yourself:

1. What does `for { a <- xs; b <- ys if a < b } yield (a, b)` desugar to?
2. Why does a sealed trait make adding a new event type safer?
3. Why is `ds.filter(_.status == "PAID")` slower than `ds.filter($"status" === "PAID")`?
4. Why does a `var` incremented inside `rdd.foreach` stay zero on the driver?